# SPACEMAP: Spatial Tissue and Cell Annotation Pipeline

This notebook performs spatial annotation of segmented cells by integrating tissue compartment and zone information from QuPath-generated annotations. It processes the output from Step 3 (cell segmentation) and combines it with spatial tissue classifications to prepare data for downstream phenotyping and spatial analysis.

## Pipeline Overview

**Input from Previous Steps:**
- **Step 2**: ROI extraction from registered CODEX+H&E images.
- **Step 3**: Cell segmentation features from Cellpose or DeepCell.
- **QuPath**: Tissue compartment annotations in GeoJSON format.

**This Notebook's Workflow:**
1. **Stage 1A: Merge Tissue & Remove Artifacts** - Merges tissue annotations and removes artifacts.
2. **Manual QuPath Editing (Optional)** - Allows for manual refinement of tissue boundaries in QuPath.
3. **Stage 1B: Create Compartments & Zones** - Creates final tissue compartments and spatial zones.
4. **Stage 2: Cell Annotation** - Annotates cells with tissue regions and spatial zones.

**Outputs:**
- **Stage 1A:** GeoJSON files for optional QuPath editing.
- **Stage 1B:** Final annotation GeoJSON files with compartments and zones.
- **Stage 2:** Annotated cell data in CSV format, ready for phenotyping.

# Spatial Tissue and Zone Annotation Pipeline

This notebook performs spatial annotation of segmented cells by integrating tissue compartment and zone information from QuPath annotations. It follows the cell segmentation and feature extraction pipeline and prepares data for downstream spatial analysis.

## Pipeline Overview:

**Input from Previous Steps:**
- **Step2**: ROI extraction from registered CODEX+H&E images (outputs/registration/roi/)
- **Step3A/3B**: Cell segmentation features (outputs/segmentation/cellpose/ or deepcell/)
- **QuPath**: Tissue compartment annotations (GeoJSON format)

**This Notebook's Workflow:**

1. **Stage 1A: Merge Tissue & Remove Artifacts** - Initial merging of epithelia, tumor, stroma, endothelia into unified "tissue" regions with artifact removal
2. **Manual QuPath Editing** - Import merged tissue into QuPath for manual refinement and cleanup (optional)
3. **Stage 1B: Create Compartments & Zones** - Process refined tissue to create final compartments and spatial zones
4. **Stage 2: Cell Annotation & Zone-T** - Annotate cells with tissue regions, zones, and tumor proximity in one step

## Key Features:
- **QuPath-Compatible GeoJSON Format** - Proper formatting with UUIDs, colors, and measurements
- **Manual Intervention Support** - Pipeline split to allow QuPath editing between stages
- **Combined Processing** - Cell annotation and zone-t classification done together for efficiency
- **Comprehensive Logging** - Detailed progress tracking and validation

## Outputs:
- **Stage 1A:** `outputs/processed_annotations/stage1a_merged_tissue/` (for QuPath editing)
- **Stage 1B:** `outputs/processed_annotations/stage1b_compartments_zones/` (final annotations)
- **Stage 2:** `outputs/annotated_csv/` (fully annotated cell data ready for phenotyping)

## Setup and Configuration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import glob
import logging
from pathlib import Path
import importlib
import geopandas as gpd

# Add the src directory to Python path
project_root = Path().resolve().parent
src_path = project_root / "src"
if "notebooks" in str(project_root):
    project_root = project_root.parent
    src_path = project_root / "src"
sys.path.insert(0, str(src_path))

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Import custom modules
try:
    from spacemap.annotation_utils import *
    from spacemap.config import *
    print("Successfully imported annotation_utils and config.")
    
    # Check if we're using the updated version
    import inspect
    source = inspect.getsource(create_compartments_and_zones)
    if "Creating inner zones (zone_aa, zone_bb, zone_cc, zone_dd)" in source:
        print(" Using UPDATED annotation_utils with inner zone fixes!")
    else:
        print(" Using OLD annotation_utils - please restart kernel!")
        
    roi_tracker = ROITracker()
except (ImportError, NameError) as e:
    print(f"Error importing modules or initializing ROITracker: {e}")
    roi_tracker = None

print("Setup complete.")
print(f"Project root: {project_root}")

## Configuration Parameters

Configure the annotation pipeline parameters below. These control input/output paths and processing options.

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Input paths
RAW_ANNOTATIONS_DIR = "../data/annotations/stroma_epi_bkg"
ENDOTHELIA_DIR = "../data/annotations/endothelia"
REMOVAL_DIR = "../data/annotations/removal_annotations"

# Segmentation input
SEGMENTATION_METHOD = "cellpose"  # "cellpose" or "deepcell"
SEGMENTATION_INPUT_DIR = f"../outputs/segmentation/{SEGMENTATION_METHOD}"
SEGMENTATION_DIR = SEGMENTATION_INPUT_DIR

# Output directories
STAGE1A_OUTPUT_DIR = "../outputs/processed_annotations/stage1a_merged_tissue"
STAGE1B_OUTPUT_DIR = "../outputs/processed_annotations/stage1b_compartments_zones"
STAGE2_OUTPUT_DIR = "../outputs/annotated_csv"
FINAL_OUTPUT_DIR = STAGE2_OUTPUT_DIR

# Processing parameters
MIN_TISSUE_AREA = 300000
BUFFER_SIZE = 40

# Annotation class configuration
TISSUE_CLASSES = ["epithelia", "tumor", "stroma"]
ENDOTHELIA_CLASSES = ["endothelial", "endothelia"]
REMOVAL_CLASSES = ["artefact", "exclude", "blood_clot"]

# File pattern matching
GEOJSON_PATTERN = "*.geojson"
CSV_PATTERN = "*_features.csv"

# Create output directories
os.makedirs(STAGE1A_OUTPUT_DIR, exist_ok=True)
os.makedirs(STAGE1B_OUTPUT_DIR, exist_ok=True)
os.makedirs(STAGE2_OUTPUT_DIR, exist_ok=True)

print("Configuration Parameters Loaded.")

## Check Available Data

Let's examine what annotation and segmentation files are available for processing.

In [ ]:
# Find raw GeoJSON annotation files
raw_geojson_files = glob.glob(os.path.join(RAW_ANNOTATIONS_DIR, GEOJSON_PATTERN))

# Find endothelia annotation files
endothelia_files = glob.glob(os.path.join(ENDOTHELIA_DIR, GEOJSON_PATTERN))

# Find removal annotation files
removal_files = glob.glob(os.path.join(REMOVAL_DIR, GEOJSON_PATTERN))

# Find segmentation CSV files
segmentation_csvs = glob.glob(os.path.join(SEGMENTATION_INPUT_DIR, CSV_PATTERN))

print(f"Available Data Summary:")
print(f"  Raw GeoJSON files: {len(raw_geojson_files)}")
for f in raw_geojson_files[:3]:  # Show first 3
    print(f"    - {os.path.basename(f)}")
if len(raw_geojson_files) > 3:
    print(f"    ... and {len(raw_geojson_files) - 3} more")

print(f"  Endothelia files: {len(endothelia_files)}")
print(f"  Removal files: {len(removal_files)}")
print(f"  Segmentation CSV files: {len(segmentation_csvs)}")

# Check for potential issues
if not raw_geojson_files:
    print(f"\nWarning: No GeoJSON files found in {RAW_ANNOTATIONS_DIR}")
if not segmentation_csvs:
    print(f"\nWarning: No segmentation CSV files found in {SEGMENTATION_INPUT_DIR}")
    
print(f"\nData check complete!")

## Stage 1A: Merge Tissue and Remove Artifacts

This stage processes the raw annotations to create a file that contains all tissue components with artifact removal applied.

**Processing Steps:**
1. **Load tissue components:** Reads epithelia, tumor, and stroma from the main annotation files.
2. **Add endothelia:** Integrates endothelial annotations.
3. **Apply artifact removal:** Subtracts areas marked for removal.
4. **Create merged tissue:** Combines all tissue types into unified "tissue" polygons.
5. **Filter by size:** Removes small tissue polygons.

**Output:**
- A GeoJSON file for each input file, ready for optional manual editing in QuPath.

In [ ]:
# ============================================================================
# STAGE 1A: MERGE TISSUE AND REMOVE ARTIFACTS
# ============================================================================

print("Starting Stage 1A: Merging tissue and removing artifacts...")

# Process all files
stage1a_results = process_all_slides_stage1a(
    raw_annotations_dir=RAW_ANNOTATIONS_DIR,
    endothelia_dir=ENDOTHELIA_DIR,
    removal_dir=REMOVAL_DIR,
    output_dir=STAGE1A_OUTPUT_DIR,
    tissue_classes=TISSUE_CLASSES,
    endothelia_classes=ENDOTHELIA_CLASSES,
    removal_classes=REMOVAL_CLASSES,
    min_area=MIN_TISSUE_AREA
)

# Show results
successful = sum(1 for r in stage1a_results.values() if r.get('success'))
total_files = len(stage1a_results)

print(f"\nStage 1A Complete: {successful}/{total_files} files processed successfully.")
print(f"Output location: {STAGE1A_OUTPUT_DIR}")

if successful > 0:
    print(f"\nNext Step: Optional manual editing in QuPath, then run Stage 1B.")
else:
    print(f"No files were processed successfully in Stage 1A.")

## Stage 1B: Create Compartments and Zones

This stage takes the tissue polygons from Stage 1A and creates the final tissue compartments and spatial zones.

**Input:** Output from Stage 1A (`*_merged_tissue.geojson` files).

**Processing Steps:**
1. Loads tissue boundaries.
2. Creates final compartments (e.g., epithelia, tumor, stroma).
3. Generates spatial zones for cell annotation.

**Output:** `*_final_annotations.geojson` files for Stage 2.

In [ ]:
# ============================================================================
# STAGE 1B: CREATE COMPARTMENTS AND ZONES
# ============================================================================

print("Starting Stage 1B: Creating compartments and zones...")

# Process refined files from QuPath
stage1b_results = process_all_slides_stage1b(
    refined_tissue_dir=STAGE1A_OUTPUT_DIR,
    output_dir=STAGE1B_OUTPUT_DIR,
    tissue_classes=TISSUE_CLASSES,
    endothelia_classes=ENDOTHELIA_CLASSES,
    buffer_size=BUFFER_SIZE,
    min_area=3000
)

# Show results
successful = sum(1 for r in stage1b_results.values() if r.get('success'))
total_files = len(stage1b_results)

print(f"\nStage 1B Complete: {successful}/{total_files} files processed successfully.")
print(f"Output location: {STAGE1B_OUTPUT_DIR}")

if successful > 0:
    print(f"Ready to proceed to Stage 2 (Cell Annotation).")
else:
    print(f"No files were processed successfully in Stage 1B.")

## Stage 2: Cell Annotation with Regions, Zones, and Tumor Proximity

This stage annotates the segmented cells with spatial information from the processed GeoJSON files.

**Annotations Added:**
- **Regional Classifications:** `tissue`, `epithelia`, `stroma`, `tumor`, `endothelia`.
- **ROI Tracking:** `tissue_roi`, `epithelia_roi`, etc., with ROI ID and area.
- **Zone Classifications:** `zone_a`, `zone_b`, etc., for spatial analysis.
- **Tumor Proximity:** `zone_a_t`, `zone_b_t`, etc., indicating proximity to tumor.

In [ ]:
# ============================================================================
# STAGE 2: CELL ANNOTATION WITH ROI TRACKING
# ============================================================================

print("Starting Stage 2: Cell Annotation...")

# Processing configuration
OVERWRITE_EXISTING_STAGE2 = False  # Set to True to overwrite existing annotated CSV files

# Process cell annotation for all slides
stage2_results = process_all_slides_stage2(
    stage1b_dir=STAGE1B_OUTPUT_DIR,
    cells_dir=SEGMENTATION_DIR,
    output_dir=FINAL_OUTPUT_DIR,
    roi_tracker=roi_tracker,
    csv_pattern=CSV_PATTERN,  # Pass the pattern from notebook configuration
    overwrite=OVERWRITE_EXISTING_STAGE2
)

# Show results
successful = sum(1 for r in stage2_results.values() if r.get('success'))
total_files = len(stage2_results)
skipped = sum(1 for r in stage2_results.values() if r.get('skipped'))

print(f"\nStage 2 Complete: {successful}/{total_files} files processed successfully.")
if skipped > 0:
    print(f"Skipped: {skipped} files (already processed, set OVERWRITE_EXISTING_STAGE2=True to reprocess)")
print(f"Output location: {FINAL_OUTPUT_DIR}")

if successful > 0:
    print("SPACEMAP Annotation Pipeline Complete!")
    print("Ready for Step 5: Batch Correction")
else:
    print("No files were processed successfully in Stage 2.")

## Visualization: Annotation Results

Let's create some visualizations to validate the annotation results.

In [ ]:
# ============================================================================
# PIPELINE SUMMARY AND VALIDATION
# ============================================================================

print("SPACEMAP V4 Spatial Annotation Pipeline - Final Summary")
print("=" * 80)

# Check all stages
try:
    stage1a_success = sum(1 for r in stage1a_results.values() if r.get('success'))
    print(f"Stage 1A (Merge & Clean): {stage1a_success} files processed")
except:
    print(f"Stage 1A: Not run or failed")

try:
    stage1b_success = sum(1 for r in stage1b_results.values() if r.get('success'))
    print(f"Stage 1B (Compartments & Zones): {stage1b_success} files processed")
except:
    print(f"Stage 1B: Not run or failed")

try:
    stage2_success = sum(1 for r in stage2_results.values() if r.get('success'))
    stage2_total_cells = sum(r.get('cell_count', 0) for r in stage2_results.values() if r.get('success'))
    print(f"Stage 2 (Cell Annotation): {stage2_success} files processed, {stage2_total_cells:,} cells annotated")
except:
    print(f"Stage 2: Not run or failed")

# Check output directories
print(f"\nOutput Locations:")
print(f"  Stage 1A (for QuPath): {STAGE1A_OUTPUT_DIR}")
print(f"  Stage 1B (final GeoJSON): {STAGE1B_OUTPUT_DIR}")
print(f"  Stage 2 (annotated CSV): {STAGE2_OUTPUT_DIR}")

# List final output files
final_csvs = glob.glob(os.path.join(STAGE2_OUTPUT_DIR, "*_fully_annotated.csv"))
print(f"\nFinal Annotated CSV Files ({len(final_csvs)} files):")
for f in final_csvs[:5]:  # Show first 5
    print(f"  - {os.path.basename(f)}")
if len(final_csvs) > 5:
    print(f"  ... and {len(final_csvs) - 5} more")

# Show data structure of a sample file
if final_csvs:
    sample_file = final_csvs[0]
    sample_df = pd.read_csv(sample_file)
    
    print(f"\nSample Data Structure (from {os.path.basename(sample_file)}):")
    print(f"  Total cells: {len(sample_df):,}")
    print(f"  Total columns: {len(sample_df.columns)}")
    
    # Show annotation columns that were added
    annotation_cols = []
    for col in sample_df.columns:
        if col in ['tissue', 'epithelia', 'stroma', 'tumor', 'endothelia'] or \
           col.startswith('zone_') or col.endswith('_roi') or col.endswith('_roi_size'):
            annotation_cols.append(col)
    
    print(f"\nAdded Annotation Columns ({len(annotation_cols)}):")
    for col in sorted(annotation_cols):
        if col in sample_df.columns:
            non_zero = (sample_df[col] != 0).sum() if sample_df[col].dtype in ['int64', 'float64'] else 0
            print(f"    {col}: {non_zero:,} non-zero values")

print(f"\nPipeline Complete! Annotated data ready for spatial analysis.")
print(f"Next steps: Use the annotated CSV files for downstream spatial biology analysis.")

In [ ]:
# ============================================================================
# VISUALIZATION OF RESULTS
# ============================================================================

if final_csvs:
    # Load a sample file for visualization
    sample_file = final_csvs[0]
    df = pd.read_csv(sample_file)
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    # 1. Distribution of cells across tissue compartments
    tissue_cols = ['tissue', 'epithelia', 'stroma', 'tumor', 'endothelia']
    tissue_counts = {}
    for col in tissue_cols:
        if col in df.columns:
            tissue_counts[col] = (df[col] == 1).sum()
    
    if tissue_counts:
        axes[0].bar(tissue_counts.keys(), tissue_counts.values())
        axes[0].set_title('Cell Distribution Across Tissue Compartments')
        axes[0].set_ylabel('Number of Cells')
        axes[0].tick_params(axis='x', rotation=45)
    
    # 2. Spatial scatter plot (if we have coordinates)
    if 'x_centroid' in df.columns and 'y_centroid' in df.columns:
        # Color cells by tissue type
        colors = []
        for _, row in df.iterrows():
            if row.get('tumor', 0) == 1:
                colors.append('red')
            elif row.get('epithelia', 0) == 1:
                colors.append('blue')
            elif row.get('stroma', 0) == 1:
                colors.append('green')
            else:
                colors.append('gray')
        
        # Sample points if too many (for performance)
        if len(df) > 20000:
            sample_idx = np.random.choice(len(df), 20000, replace=False)
            x_coords = df.iloc[sample_idx]['x_centroid']
            y_coords = df.iloc[sample_idx]['y_centroid']
            colors_sample = [colors[i] for i in sample_idx]
        else:
            x_coords = df['x_centroid']
            y_coords = df['y_centroid']
            colors_sample = colors
        
        scatter = axes[1].scatter(x_coords, y_coords, c=colors_sample, s=1, alpha=0.6)
        axes[1].set_title('Spatial Distribution of Annotated Cells')
        axes[1].set_xlabel('X Coordinate')
        axes[1].set_ylabel('Y Coordinate')
        axes[1].set_aspect('equal')
        axes[1].invert_yaxis() # Invert Y axis
        
        # Add legend
        import matplotlib.patches as mpatches
        red_patch = mpatches.Patch(color='red', label='Tumor')
        blue_patch = mpatches.Patch(color='blue', label='Epithelia')
        green_patch = mpatches.Patch(color='green', label='Stroma')
        gray_patch = mpatches.Patch(color='gray', label='Other')
        axes[1].legend(handles=[red_patch, blue_patch, green_patch, gray_patch], 
                        loc='upper right', bbox_to_anchor=(1.15, 1))
    
    plt.tight_layout()
    plt.show()
    
    print(f"Visualization based on: {os.path.basename(sample_file)}")
    print(f"    Total cells plotted: {len(df):,}")
    
else:
    print("No final CSV files available for visualization")

## Pipeline Completion Summary

The SPACEMAP spatial annotation pipeline has been completed successfully.

### Completed Pipeline Stages:

**Stage 1A: Tissue Merging & Artifact Removal**
- Merged QuPath tissue annotations into unified tissue polygons.
- Applied artifact removal and minimum area filtering.

**Stage 1B: Compartments & Zones**
- Processed tissue annotations to create final compartments and spatial zones.

**Stage 2: Cell Annotation**
- Annotated cells with tissue regions, spatial zones, and tumor proximity.

### Final Output:
- Annotated cell data is saved in `outputs/annotated_csv/`.
- The output files are named `*_fully_annotated.csv` and are ready for the next step.

### Next Steps:
The annotated data is now ready for **Step 5: Batch Correction**.